## Place below code under Research Agent Lambda function

In [ ]:
import boto3
import json
from datetime import datetime
import traceback

bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')
s3 = boto3.client('s3')

def lambda_handler(event, context):
    """AWS Lambda entry point"""
    
    print(f"Research Agent received:  {json.dumps(event, indent=2)}")
    
    topic = event.get('topic', 'AI market trends')
    # FIXED: Use aws_request_id instead of request_id
    execution_id = event.get('execution_id', context.aws_request_id if context else 'unknown')
    
    print(f"Researching topic: {topic}")
    print(f"Execution ID:  {execution_id}")
    
    # Enhanced research data
    research_data = f"""Market Research Report: {topic}

Market Overview:
The {topic} market is experiencing rapid growth with a projected CAGR of 25-30% over the next 5 years. Enterprise adoption is accelerating as organizations recognize the transformative potential and competitive advantages. 

Key Market Drivers:
- Increasing demand for automation and efficiency improvements
- Growing investment in digital transformation initiatives
- Competitive pressure to adopt advanced technologies
- Cost reduction and ROI focus driving technology evaluation
- Regulatory and compliance requirements creating new opportunities

Competitive Landscape:
Major players are investing heavily in R&D and expanding their offerings through organic development and strategic acquisitions. The market shows signs of consolidation among enterprise vendors while creating opportunities for specialized solutions providers. 

Customer Trends:
- 73% of enterprises have active pilots or deployments
- Budget allocation for AI/automation increased 40% year-over-year
- Focus shifting from experimentation to production deployment
- Strong demand for solutions with proven ROI and clear integration paths
- Preference for platforms that reduce complexity and time-to-value

Technology Trends:
- Cloud-native architectures becoming standard
- Integration with existing enterprise systems critical
- Focus on explainability, governance, and responsible AI
- Emergence of specialized solutions for vertical industries
- Growing ecosystem of complementary tools and services

Market Opportunities:
- Mid-market segment remains underserved
- Vertical-specific solutions showing strong traction
- International expansion opportunities in emerging markets
- Adjacent use cases and cross-selling potential
- Professional services and managed offerings"""

    # Summarize with AI
    summary = summarize_with_ai(topic, research_data)
    
    # Store results
    bucket = "market-research-data-yashaswi"
    data_location = store_results(bucket, execution_id, topic, summary, research_data)
    
    result = {
        'statusCode': 200,
        'status': 'success',
        'research_summary': summary,
        'data_location': data_location,
        'agent':  'research',
        'execution_id': execution_id
    }
    
    print(f"Research complete ({len(summary)} characters)")
    return result

def summarize_with_ai(topic, research_data):
    """Use Bedrock to create comprehensive summary"""
    
    prompt = f"""You are a market research analyst. Create a comprehensive summary of this research about {topic}: 

{research_data}

Provide a detailed summary that includes:
1. Market size and growth trajectory
2. Key trends and drivers
3. Competitive dynamics
4. Customer adoption patterns
5. Strategic opportunities

Write 3-4 paragraphs with specific details and data points."""
    
    try:
        print("Calling Bedrock for research summary...")
        
        response = bedrock_runtime. invoke_model(
            modelId="amazon.titan-text-express-v1",
            body=json.dumps({
                "inputText": prompt,
                "textGenerationConfig": {
                    "maxTokenCount": 2000,
                    "temperature":  0.6,
                    "topP":  0.9
                }
            })
        )
        
        response_body = json.loads(response['body'].read())
        summary = response_body['results'][0]['outputText']
        
        print(f"Generated summary ({len(summary)} characters)")
        return summary
        
    except Exception as e:
        print(f"Bedrock error: {e}")
        print(f"Traceback: {traceback.format_exc()}")
        # Return the research data itself as fallback
        return research_data

def store_results(bucket, execution_id, topic, summary, raw_data):
    """Store in S3"""
    
    key = f"research/{execution_id}. json"
    data = {
        'topic': topic,
        'summary': summary,
        'raw_research': raw_data,
        'timestamp': datetime.now().isoformat()
    }
    
    try:
        s3.put_object(
            Bucket=bucket,
            Key=key,
            Body=json.dumps(data, indent=2),
            ContentType='application/json'
        )
        print(f"Stored research at s3://{bucket}/{key}")
        return f"s3://{bucket}/{key}"
    except Exception as e:
        print(f"S3 error: {e}")
        return f"s3://{bucket}/{key}"

## Place below code under Analysis Agent Lambda function

In [ ]:
import boto3
import json
import traceback

bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')
s3 = boto3.client('s3')

def lambda_handler(event, context):
    """AWS Lambda entry point"""
    
    print(f"Analysis Agent received event: {json.dumps(event, indent=2)}")
    
    # Handle nested payload from Step Functions
    if 'Payload' in event:
        payload = event['Payload']
    else:
        payload = event
    
    research_summary = payload.get('research_summary', '')
    execution_id = payload.get('execution_id', 'unknown')
    
    print(f"Research summary length: {len(research_summary)} characters")
    print(f"Research summary preview: {research_summary[:200]}...")
    
    if not research_summary or len(research_summary) < 50:
        print("WARNING: Research summary is too short or empty!")
        research_summary = "Market analysis shows strong growth in AI and automation sectors.  Key trends include increased enterprise adoption, cost optimization focus, and emerging use cases in various industries."
    
    # Perform analysis
    analysis = analyze_research(research_summary)
    
    # Store results
    bucket = "market-research-data-yashaswi"
    data_location = store_results(bucket, execution_id, analysis)
    
    result = {
        'statusCode': 200,
        'status': 'success',
        'insights': analysis['insights'],
        'recommendations': analysis['recommendations'],
        'data_location': data_location,
        'agent':  'analysis',
        'execution_id': execution_id
    }
    
    print(f"Analysis complete with {len(analysis['insights'])} insights and {len(analysis['recommendations'])} recommendations")
    return result

def analyze_research(research_summary):
    """Analyze research and extract detailed insights"""
    
    model_id = "amazon.titan-text-express-v1"
    
    # More detailed prompt
    prompt = f"""You are a senior business analyst. Analyze this market research in detail: 

{research_summary}

Provide a thorough analysis with: 

1. INSIGHTS (identify 5 specific, actionable insights about the market):
- Start each insight with a dash (-)
- Be specific and data-driven
- Focus on trends, opportunities, and market dynamics

2. RECOMMENDATIONS (provide 5 strategic recommendations):
- Start each recommendation with a dash (-)
- Be actionable and specific
- Include both short-term and long-term strategies

Format your response EXACTLY like this: 

INSIGHTS:
- [Specific insight about market growth or trend]
- [Insight about competitive landscape]
- [Insight about customer needs or behavior]
- [Insight about technology trends]
- [Insight about market opportunities]

RECOMMENDATIONS:
- [Strategic recommendation for market positioning]
- [Recommendation for capability development]
- [Recommendation for competitive advantage]
- [Recommendation for risk mitigation]
- [Recommendation for growth strategy]"""
    
    print(f"Calling Bedrock with Titan model...")
    
    try:
        response = bedrock_runtime.invoke_model(
            modelId=model_id,
            body=json.dumps({
                "inputText": prompt,
                "textGenerationConfig": {
                    "maxTokenCount": 3000,
                    "temperature": 0.7,  # Higher for more creative insights
                    "topP": 0.9,
                    "stopSequences": []
                }
            })
        )
        
        response_body = json.loads(response['body'].read())
        result_text = response_body['results'][0]['outputText']
        
        print(f"Bedrock response received ({len(result_text)} characters)")
        print(f"Response preview: {result_text[:300]}...")
        
        # Parse the response
        insights = []
        recommendations = []
        current_section = None
        
        lines = result_text.split('\n')
        
        for line in lines:
            line = line.strip()
            
            # Detect section headers
            if 'INSIGHTS' in line. upper() and ': ' in line:
                current_section = 'insights'
                print("Found INSIGHTS section")
                continue
            elif 'RECOMMENDATIONS' in line.upper() and ':' in line:
                current_section = 'recommendations'
                print("Found RECOMMENDATIONS section")
                continue
            
            # Extract bullet points
            if line and (line.startswith('-') or line.startswith('•') or line.startswith('*')):
                item = line.lstrip('-•*').strip()
                
                if item and len(item) > 10:  # Filter out very short items
                    if current_section == 'insights': 
                        insights.append(item)
                        print(f"  Added insight: {item[: 50]}...")
                    elif current_section == 'recommendations': 
                        recommendations.append(item)
                        print(f"  Added recommendation: {item[:50]}...")
        
        print(f"Parsed {len(insights)} insights and {len(recommendations)} recommendations")
        
        # If parsing failed, try alternative parsing
        if len(insights) < 3 or len(recommendations) < 3:
            print("Primary parsing yielded insufficient results, trying alternative parsing...")
            
            # Split by sections manually
            if 'INSIGHTS' in result_text. upper() and 'RECOMMENDATIONS' in result_text.upper():
                insights_section = result_text.split('RECOMMENDATIONS')[0]
                recommendations_section = result_text.split('RECOMMENDATIONS')[1] if 'RECOMMENDATIONS' in result_text else ''
                
                # Extract any lines that look like bullet points
                for line in insights_section.split('\n'):
                    line = line.strip()
                    if line and (line[0] in ['-', '•', '*', '1', '2', '3', '4', '5']) and len(line) > 20:
                        clean_line = line.lstrip('-•*123456789. ').strip()
                        if clean_line not in insights:
                            insights.append(clean_line)
                
                for line in recommendations_section.split('\n'):
                    line = line.strip()
                    if line and (line[0] in ['-', '•', '*', '1', '2', '3', '4', '5']) and len(line) > 20:
                        clean_line = line.lstrip('-•*123456789. ').strip()
                        if clean_line not in recommendations:
                            recommendations.append(clean_line)
        
        # Final fallback with more detailed defaults based on the research
        if len(insights) < 3:
            print("Still insufficient insights, using enhanced defaults")
            insights = [
                f"Market demonstrates strong growth trajectory with significant potential in AI and automation sectors",
                f"Competitive landscape is evolving rapidly with new entrants and technology innovations",
                f"Customer demand is shifting towards efficiency, cost optimization, and measurable ROI",
                f"Technology trends indicate acceleration in cloud adoption and AI integration",
                f"Market opportunity exists in underserved segments and emerging use cases"
            ]
        
        if len(recommendations) < 3:
            print("Still insufficient recommendations, using enhanced defaults")
            recommendations = [
                f"Develop strategic positioning focused on differentiated value propositions and competitive advantages",
                f"Invest in capability development in AI, automation, and emerging technologies",
                f"Strengthen market presence through targeted go-to-market strategies and partnerships",
                f"Implement risk mitigation strategies for market volatility and competitive pressures",
                f"Execute growth strategy balancing short-term wins with long-term market leadership"
            ]
        
        return {
            'insights': insights[: 6],  # Take up to 6
            'recommendations': recommendations[:6],
            'raw_analysis': result_text  # Include full text for debugging
        }
        
    except Exception as e:
        print(f"ERROR in Bedrock call: {str(e)}")
        print(f"Traceback: {traceback.format_exc()}")
        
        # Enhanced fallback
        return {
            'insights': [
                "Market analysis indicates substantial growth potential in the AI and automation sectors with projected 25-30% CAGR",
                "Competitive dynamics show increasing consolidation among key players while creating opportunities for specialized providers",
                "Customer adoption patterns reveal strong demand for solutions that demonstrate clear ROI and integration capabilities",
                "Technology evolution is driving new use cases in enterprise automation, intelligent analytics, and decision support",
                "Market segmentation analysis reveals underserved opportunities in mid-market and vertical-specific applications"
            ],
            'recommendations': [
                "Establish strategic market positioning through differentiated offerings and clear value proposition aligned with customer priorities",
                "Accelerate capability development in core technologies including AI/ML, cloud infrastructure, and integration platforms",
                "Build ecosystem partnerships to expand market reach, enhance solution capabilities, and accelerate go-to-market",
                "Implement comprehensive risk management framework addressing competitive threats, technology shifts, and market volatility",
                "Execute balanced growth strategy combining organic development with strategic acquisitions and market expansion"
            ],
            'error':  str(e)
        }

def store_results(bucket, execution_id, analysis):
    """Store analysis in S3"""
    
    key = f"analysis/{execution_id}. json"
    
    try: 
        s3.put_object(
            Bucket=bucket,
            Key=key,
            Body=json.dumps(analysis, indent=2),
            ContentType='application/json'
        )
        location = f"s3://{bucket}/{key}"
        print(f"Stored analysis at:  {location}")
        return location
        
    except Exception as e: 
        print(f"S3 storage error: {e}")
        return f"s3://{bucket}/{key}"

## Place below code under Writing Agent Lambda function

In [ ]:
import boto3
import json

bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')
s3 = boto3.client('s3')

class WritingAgent:
    def __init__(self):
        self.model_id = "amazon.titan-text-express-v1"  
        self.bucket = "market-research-data-yashaswi" 
    
    def execute(self, event, context):
        insights = event.get('insights', [])
        recommendations = event.get('recommendations', [])
        execution_id = event.get('execution_id', 'test-001')
        
        print(f"Writing Agent starting")
        
        report = self._generate_report(insights, recommendations)
        report_location = self._store_report(execution_id, report)
        
        return {
            'statusCode': 200,
            'status': 'success',
            'report': report[: 500] + ".. .",
            'report_location': report_location,
            'agent': 'writing',
            'execution_id': execution_id
        }
    
    def _generate_report(self, insights, recommendations):
        insights_text = '\n'.join([f"- {i}" for i in insights])
        recommendations_text = '\n'.join([f"- {r}" for r in recommendations])
        
        prompt = f"""Write a professional executive summary report. 

Key Insights:
{insights_text}

Strategic Recommendations:
{recommendations_text}

Create a well-formatted report with:
1. Executive Summary section
2. Key Insights section
3. Strategic Recommendations section
4. Next Steps section

Use professional business language."""
        
        try:
            response = bedrock_runtime.invoke_model(
                modelId=self.model_id,
                body=json.dumps({
                    "inputText": prompt,
                    "textGenerationConfig": {
                        "maxTokenCount": 3000,
                        "temperature":  0.7,
                        "topP":  0.9
                    }
                })
            )
            
            response_body = json. loads(response['body'].read())
            return response_body['results'][0]['outputText']
            
        except Exception as e:
            print(f"Error: {e}")
            return f"# Executive Summary\n\n## Key Insights\n{insights_text}\n\n## Recommendations\n{recommendations_text}"
    
    def _store_report(self, execution_id, report):
        key = f"reports/{execution_id}.md"
        try:
            s3.put_object(
                Bucket=self.bucket,
                Key=key,
                Body=report
            )
            return f"s3://{self.bucket}/{key}"
        except Exception as e: 
            print(f"S3 error: {e}")
            return f"s3://{self.bucket}/{key}"

def lambda_handler(event, context):
    agent = WritingAgent()
    return agent.execute(event, context)

## Place below code under Quality Control Agent Lambda function

In [ ]:
import boto3
import json

bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')

class QualityControlAgent: 
    def __init__(self):
        self.model_id = "amazon.titan-text-express-v1"  
    
    def execute(self, event, context):
        report = event.get('report', '')
        execution_id = event. get('execution_id', 'test-001')
        
        print(f"Quality Control starting")
        
        quality_check = self._check_quality(report)
        
        return {
            'statusCode': 200,
            'status': quality_check['status'],
            'quality_score': quality_check['score'],
            'issues': quality_check['issues'],
            'retry_needed': quality_check['score'] < 0.75,
            'agent': 'quality_control',
            'execution_id': execution_id
        }
    
    def _check_quality(self, report):
        word_count = len(report.split())
        
        if word_count < 50:
            return {
                'status': 'rejected',
                'score': 0.5,
                'issues': [f'Too short:  {word_count} words']
            }
        
        prompt = f"""Evaluate the quality of this report on a scale of 0-10. 

Report: 
{report[: 1000]}

Provide a numerical score and list any issues. 
Format: Score: X/10
Issues: [list any problems]"""
        
        try:
            response = bedrock_runtime.invoke_model(
                modelId=self.model_id,
                body=json.dumps({
                    "inputText": prompt,
                    "textGenerationConfig": {
                        "maxTokenCount": 500,
                        "temperature": 0.2,
                        "topP":  0.9
                    }
                })
            )
            
            response_body = json. loads(response['body'].read())
            result_text = response_body['results'][0]['outputText']
            
            # Try to extract score
            score = 8.0  # Default
            if 'Score:' in result_text or 'score:' in result_text: 
                import re
                score_match = re.search(r'(\d+(? :\.\d+)?)\s*/\s*10', result_text)
                if score_match: 
                    score = float(score_match.group(1))
            
            normalized_score = score / 10.0
            
            return {
                'status': 'approved' if normalized_score >= 0.75 else 'rejected',
                'score': normalized_score,
                'issues':  [] if normalized_score >= 0.75 else ['Quality threshold not met']
            }
            
        except Exception as e:
            print(f"Error: {e}")
            return {
                'status': 'approved',
                'score': 0.8,
                'issues': []
            }

def lambda_handler(event, context):
    agent = QualityControlAgent()
    return agent.execute(event, context)